# Bengali NLP Safety Challenge 2026 — Optimized V2

## Goal

Build a strong submission CSV for the Bengali NLP Safety Challenge.

This version goes beyond the starter notebook with:

1. **Word + character TF-IDF**
2. **Per-label Logistic Regression hyperparameter tuning**
3. **Per-label probability threshold tuning**
4. **BanglaBERT fine-tuning**
5. Optional **BanglishBERT / XLM-R** model experiments
6. **Class-imbalance-aware transformer loss**
7. **Multiple transformer seeds**
8. **Probability ensembling**
9. Validation-based model/weight selection
10. Final training on all labeled data
11. Automatic generation of `submission.csv`

### Important

There is no honest way to guarantee the highest Kaggle score before running experiments on the actual competition data. This notebook is designed to search stronger configurations systematically rather than guessing one "magic" setting.

**Start with the default configuration. After it works, enable additional models/seeds if you have enough GPU time.**


# 🚀 Colab Setup — Bengali NLP Safety Challenge

Run the setup cells below first. You can upload `kaggle.json`, download the competition data, then use **Runtime → Run all**.

In [ ]:
# 1) Upload your Kaggle API key (kaggle.json)
from google.colab import files

uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Please upload the Kaggle API file named kaggle.json'

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API key configured.')

In [ ]:
# 2) Download and extract the competition data
import os, zipfile, glob

DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)

!kaggle competitions download -c bengali-nlp-safety-challenge-2026 -p /content/data

zips = glob.glob('/content/data/*.zip')
assert zips, 'Competition ZIP was not downloaded. Check your Kaggle API access.'

with zipfile.ZipFile(zips[0], 'r') as z:
    z.extractall(DATA_ROOT)

print('CSV files found:')
for p in glob.glob('/content/data/**/*.csv', recursive=True):
    print(' -', p)


# 0. Recommended strategy

The strongest practical approach here is usually:

```text
                 TRAIN
                   │
          ┌────────┴────────┐
          ▼                 ▼
     TF-IDF models      Transformer models
          │                 │
          │          ┌──────┼────────┐
          │          ▼      ▼        ▼
          │       Bangla  Banglish  XLM-R
          │       BERT     BERT
          │          │
          └──────────┴───────────────┐
                                     ▼
                           Probability ensemble
                                     │
                                     ▼
                           Per-label thresholds
                                     │
                                     ▼
                                submission.csv
```

The model that wins locally is not necessarily the model that wins Kaggle. Therefore, keep the experiments and compare Kaggle results.


In [ ]:
# ============================================================
# 1. Imports and reproducibility
# ============================================================

import os
import re
import gc
import copy
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()


In [ ]:
# ============================================================
# 2. Main configuration
# ============================================================

LABELS = [
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]

NUM_LABELS = len(LABELS)

# ---------- TF-IDF ----------
TFIDF_C_GRID = [0.5, 1.0, 2.0, 4.0, 8.0]

WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 6)

WORD_MAX_FEATURES = 200_000
CHAR_MAX_FEATURES = 250_000

# ---------- Transformer ----------
TRANSFORMER_MODELS = {
    "BanglaBERT": "csebuetnlp/banglabert",
    "BanglishBERT": "csebuetnlp/banglishbert",
    "XLM-R": "FacebookAI/xlm-roberta-base",
}

PRIMARY_MODEL = "BanglaBERT"

MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 8
VALID_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
MAX_GRAD_NORM = 1.0

# Set True only after the primary model works.
RUN_SECOND_TRANSFORMER = False
SECOND_TRANSFORMER = "BanglishBERT"

# Multiple seeds improve ensemble stability but cost more GPU time.
RUN_MULTIPLE_SEEDS = False
TRANSFORMER_SEEDS = [42, 123, 2026]

# Rare-label experiment. Start False.
USE_POS_WEIGHT = False

# Mixed precision when CUDA is available.
USE_AMP = True

# Threshold search.
THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.95
THRESHOLD_STEP = 0.01

print("Primary:", PRIMARY_MODEL)
print("Labels:", LABELS)


# 3. Find the competition files

The code supports both Kaggle and local execution.

Expected files:

```text
train.csv
validation.csv
test.csv
sample_submission.csv
```


In [ ]:
def find_data_dir():
    candidates = [
        Path("/kaggle/input/bengali-nlp-safety-challenge-2026"),
        Path("/kaggle/input"),
        Path("./data"),
        Path("."),
    ]

    for base in candidates:
        if base.is_dir():
            if (base / "train.csv").exists() and (base / "test.csv").exists():
                return base

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.iterdir():
            if p.is_dir() and (p / "train.csv").exists() and (p / "test.csv").exists():
                return p

    raise FileNotFoundError(
        "Could not find train.csv/test.csv. "
        "Set DATA_DIR manually."
    )

DATA_DIR = find_data_dir()

TRAIN_PATH = DATA_DIR / "train.csv"
VALID_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

print("DATA_DIR:", DATA_DIR)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)
test_df = pd.read_csv(TEST_PATH)

sample_submission = (
    pd.read_csv(SAMPLE_SUB_PATH)
    if SAMPLE_SUB_PATH.exists()
    else None
)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

required_train = ["ID", "text"] + LABELS
required_test = ["ID", "text"]

assert all(c in train_df.columns for c in required_train)
assert all(c in valid_df.columns for c in required_train)
assert all(c in test_df.columns for c in required_test)

display(train_df.head())


In [ ]:
# ============================================================
# 4. Text normalization
# ============================================================

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize common Unicode spacing characters.
    text = text.replace("\u200c", " ").replace("\u200d", " ")

    # Keep punctuation, emojis and spelling patterns.
    text = re.sub(r"\s+", " ", text)

    return text.strip()

for df in [train_df, valid_df, test_df]:
    df["clean_text"] = df["text"].apply(clean_text)

print(train_df["clean_text"].head().tolist())


# 5. Data diagnostics

This section helps identify class imbalance and possible data issues.

For this competition, **Macro F1** is especially important because rare labels should not disappear behind the common labels.


In [ ]:
label_stats = pd.DataFrame({
    "positive_count": train_df[LABELS].sum(),
    "positive_ratio": train_df[LABELS].mean(),
})

label_stats["positive_percent"] = 100 * label_stats["positive_ratio"]

display(label_stats.sort_values("positive_ratio"))

train_duplicates = train_df["clean_text"].duplicated().sum()
valid_duplicates = valid_df["clean_text"].duplicated().sum()

print("Duplicate train texts:", train_duplicates)
print("Duplicate validation texts:", valid_duplicates)


In [ ]:
# Label co-occurrence
co_occurrence = train_df[LABELS].T.dot(train_df[LABELS])
display(co_occurrence)


# 6. Evaluation helpers

All model selection below is based on **validation Macro F1** after optimizing a separate threshold for every label.


In [ ]:
def tune_thresholds(
    y_true,
    probabilities,
    labels=LABELS,
    start=THRESHOLD_MIN,
    stop=THRESHOLD_MAX,
    step=THRESHOLD_STEP,
):
    grid = np.arange(start, stop + 1e-9, step)
    thresholds = []
    rows = []

    for i, label in enumerate(labels):
        best_t = 0.50
        best_f1 = -1.0

        for t in grid:
            pred = (probabilities[:, i] >= t).astype(np.int8)
            score = f1_score(
                y_true[:, i],
                pred,
                zero_division=0
            )

            if score > best_f1:
                best_f1 = score
                best_t = float(t)

        thresholds.append(best_t)
        rows.append({
            "label": label,
            "threshold": best_t,
            "f1": best_f1,
        })

    return np.array(thresholds), pd.DataFrame(rows)

def multilabel_score(y_true, probabilities, thresholds):
    predictions = (
        probabilities >= thresholds.reshape(1, -1)
    ).astype(np.int8)

    return {
        "macro_f1": f1_score(
            y_true, predictions,
            average="macro",
            zero_division=0,
        ),
        "micro_f1": f1_score(
            y_true, predictions,
            average="micro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true, predictions,
            average="weighted",
            zero_division=0,
        ),
        "samples_f1": f1_score(
            y_true, predictions,
            average="samples",
            zero_division=0,
        ),
        "predictions": predictions,
    }

def per_label_scores(y_true, predictions):
    rows = []
    for i, label in enumerate(LABELS):
        rows.append({
            "label": label,
            "precision": precision_score(
                y_true[:, i], predictions[:, i],
                zero_division=0
            ),
            "recall": recall_score(
                y_true[:, i], predictions[:, i],
                zero_division=0
            ),
            "f1": f1_score(
                y_true[:, i], predictions[:, i],
                zero_division=0
            ),
        })
    return pd.DataFrame(rows)


# 7. Strong TF-IDF model

### Improvement over the starter

Instead of using the same Logistic Regression `C` for all seven labels, we search for the best `C` **independently for each label**.

That is useful because:

```text
profanity
sarcasm
threatening
identity_attack
...
```

do not necessarily have the same difficulty or feature distribution.


In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=WORD_NGRAM_RANGE,
    min_df=2,
    max_df=0.995,
    max_features=WORD_MAX_FEATURES,
    sublinear_tf=True,
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=CHAR_NGRAM_RANGE,
    min_df=2,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
)

X_train_word = word_vectorizer.fit_transform(train_df["clean_text"])
X_valid_word = word_vectorizer.transform(valid_df["clean_text"])

X_train_char = char_vectorizer.fit_transform(train_df["clean_text"])
X_valid_char = char_vectorizer.transform(valid_df["clean_text"])

X_train_tfidf = hstack(
    [X_train_word, X_train_char],
    format="csr",
)

X_valid_tfidf = hstack(
    [X_valid_word, X_valid_char],
    format="csr",
)

y_train = train_df[LABELS].to_numpy(dtype=np.int8)
y_valid = valid_df[LABELS].to_numpy(dtype=np.int8)

print("Combined train features:", X_train_tfidf.shape)


In [ ]:
# Search C independently for every label.
best_tfidf_models = {}
best_tfidf_C = {}
tfidf_valid_prob = np.zeros((len(valid_df), NUM_LABELS), dtype=np.float32)

tfidf_tuning_rows = []

for i, label in enumerate(LABELS):
    best_f1 = -1.0
    best_model = None
    best_c = None
    best_prob = None

    print(f"\n===== {label} =====")

    for C in TFIDF_C_GRID:
        clf = LogisticRegression(
            C=C,
            max_iter=3000,
            class_weight="balanced",
            solver="liblinear",
            random_state=SEED,
        )

        clf.fit(X_train_tfidf, y_train[:, i])

        prob = clf.predict_proba(X_valid_tfidf)[:, 1]

        # Temporarily optimize this label's threshold.
        local_thresholds, local_df = tune_thresholds(
            y_valid[:, [i]],
            prob.reshape(-1, 1),
            labels=[label],
        )

        local_f1 = float(local_df.iloc[0]["f1"])

        tfidf_tuning_rows.append({
            "label": label,
            "C": C,
            "threshold": float(local_thresholds[0]),
            "f1": local_f1,
        })

        if local_f1 > best_f1:
            best_f1 = local_f1
            best_model = clf
            best_c = C
            best_prob = prob

    best_tfidf_models[label] = best_model
    best_tfidf_C[label] = best_c
    tfidf_valid_prob[:, i] = best_prob

    print(
        f"Best C={best_c}, "
        f"validation F1={best_f1:.4f}"
    )

tfidf_tuning_df = pd.DataFrame(tfidf_tuning_rows)
display(
    tfidf_tuning_df.sort_values(
        ["label", "f1"],
        ascending=[True, False]
    ).round(4)
)


In [ ]:
tfidf_thresholds, tfidf_threshold_df = tune_thresholds(
    y_valid,
    tfidf_valid_prob,
)

tfidf_scores = multilabel_score(
    y_valid,
    tfidf_valid_prob,
    tfidf_thresholds,
)

print("TF-IDF Macro F1:", round(tfidf_scores["macro_f1"], 4))
print("TF-IDF Micro F1:", round(tfidf_scores["micro_f1"], 4))

display(tfidf_threshold_df.round(4))
display(
    per_label_scores(
        y_valid,
        tfidf_scores["predictions"]
    ).round(4)
)


# 8. Transformer setup

We use dynamic padding, mixed precision, gradient accumulation and early stopping.

The primary model is BanglaBERT.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("GPU:", torch.cuda.get_device_name(0))

print("Device:", DEVICE)


In [ ]:
class ToxicityDataset(Dataset):
    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length,
        labels=None,
    ):
        self.texts = (
            dataframe["clean_text"]
            .fillna("")
            .astype(str)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = (
            None
            if labels is None
            else dataframe[labels].to_numpy(dtype=np.float32)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )

        if self.labels is not None:
            item["labels"] = self.labels[idx]

        return item


# 9. Transformer training function

This function can train any of the supported models with the same pipeline.

The best checkpoint is selected using validation Macro F1 after threshold optimization.


In [ ]:
def train_transformer(
    model_name,
    seed=42,
    max_length=128,
    learning_rate=2e-5,
    epochs=5,
):
    seed_everything(seed)

    print("\n" + "=" * 80)
    print("MODEL:", model_name)
    print("SEED :", seed)
    print("MAX  :", max_length)
    print("LR   :", learning_rate)
    print("=" * 80)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )

    train_dataset = ToxicityDataset(
        train_df, tokenizer, max_length, LABELS
    )
    valid_dataset = ToxicityDataset(
        valid_df, tokenizer, max_length, LABELS
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=2 if torch.cuda.is_available() else 0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collator,
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=VALID_BATCH_SIZE,
        shuffle=False,
        num_workers=2 if torch.cuda.is_available() else 0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collator,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification",
    ).to(DEVICE)

    positive = y_train.sum(axis=0)
    negative = len(y_train) - positive

    if USE_POS_WEIGHT:
        pos_weight = np.where(
            positive > 0,
            negative / positive,
            1.0,
        )

        pos_weight_tensor = torch.tensor(
            pos_weight,
            dtype=torch.float32,
            device=DEVICE,
        )
    else:
        pos_weight_tensor = None

    if pos_weight_tensor is None:
        loss_fn = torch.nn.BCEWithLogitsLoss()
    else:
        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=pos_weight_tensor
        )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
    )

    steps_per_epoch = int(
        np.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
    )

    total_steps = steps_per_epoch * epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    amp_enabled = USE_AMP and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(
        enabled=amp_enabled
    )

    best_macro = -1.0
    best_state = None
    best_thresholds = None
    best_probs = None
    best_history = []

    patience = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        train_loss_sum = 0.0

        pbar = tqdm(
            enumerate(train_loader, start=1),
            total=len(train_loader),
            desc=f"Epoch {epoch}/{epochs}",
        )

        for step, batch in pbar:
            labels = batch.pop("labels").to(
                DEVICE, dtype=torch.float32
            )
            batch = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            with torch.cuda.amp.autocast(
                enabled=amp_enabled
            ):
                logits = model(**batch).logits
                loss = loss_fn(logits, labels)
                backward_loss = (
                    loss / GRADIENT_ACCUMULATION_STEPS
                )

            scaler.scale(backward_loss).backward()

            if (
                step % GRADIENT_ACCUMULATION_STEPS == 0
                or step == len(train_loader)
            ):
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM,
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            train_loss_sum += float(loss.item())

            pbar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        # Validation
        model.eval()

        all_probs = []
        all_true = []

        with torch.no_grad():
            for batch in tqdm(
                valid_loader,
                desc="Validation"
            ):
                labels = batch.pop("labels")

                batch = {
                    k: v.to(DEVICE)
                    for k, v in batch.items()
                }

                with torch.cuda.amp.autocast(
                    enabled=amp_enabled
                ):
                    logits = model(**batch).logits

                probs = torch.sigmoid(
                    logits.float()
                ).cpu().numpy()

                all_probs.append(probs)
                all_true.append(
                    labels.numpy()
                )

        val_prob = np.vstack(all_probs)
        val_true = np.vstack(all_true).astype(np.int8)

        thresholds, threshold_df = tune_thresholds(
            val_true,
            val_prob,
        )

        scores = multilabel_score(
            val_true,
            val_prob,
            thresholds,
        )

        train_loss = (
            train_loss_sum / len(train_loader)
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "macro_f1": scores["macro_f1"],
            "micro_f1": scores["micro_f1"],
            "weighted_f1": scores["weighted_f1"],
            "samples_f1": scores["samples_f1"],
        }

        best_history.append(row)

        print(
            f"Epoch {epoch}: "
            f"loss={train_loss:.4f} | "
            f"macro={scores['macro_f1']:.4f} | "
            f"micro={scores['micro_f1']:.4f}"
        )

        if scores["macro_f1"] > best_macro:
            best_macro = scores["macro_f1"]
            best_state = copy.deepcopy(
                model.state_dict()
            )
            best_thresholds = thresholds.copy()
            best_probs = val_prob.copy()
            patience = 0

            print("✓ New best checkpoint")

        else:
            patience += 1

            if patience >= EARLY_STOPPING_PATIENCE:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    model.eval()

    del train_loader, valid_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "model": model,
        "tokenizer": tokenizer,
        "best_macro_f1": best_macro,
        "valid_prob": best_probs,
        "thresholds": best_thresholds,
        "history": pd.DataFrame(best_history),
        "model_name": model_name,
        "seed": seed,
    }


# 10. Train the primary model

Start here.

If this works and gives a good validation Macro F1, then enable additional models/seeds later.


In [ ]:
primary_result = train_transformer(
    model_name=TRANSFORMER_MODELS[PRIMARY_MODEL],
    seed=SEED,
    max_length=MAX_LENGTH,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
)

print(
    "Best",
    PRIMARY_MODEL,
    "Macro F1:",
    round(primary_result["best_macro_f1"], 4)
)

display(primary_result["history"].round(4))


In [ ]:
primary_valid_prob = primary_result["valid_prob"]
primary_thresholds = primary_result["thresholds"]

primary_scores = multilabel_score(
    y_valid,
    primary_valid_prob,
    primary_thresholds,
)

display(
    per_label_scores(
        y_valid,
        primary_scores["predictions"]
    ).round(4)
)


# 11. Optional second transformer

BanglishBERT is worth testing if the dataset contains Banglish / code-mixed text.

Turn `RUN_SECOND_TRANSFORMER = True` only after the primary model works.


In [ ]:
second_result = None

if RUN_SECOND_TRANSFORMER:
    second_result = train_transformer(
        model_name=TRANSFORMER_MODELS[SECOND_TRANSFORMER],
        seed=SEED,
        max_length=MAX_LENGTH,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
    )

    print(
        "Best",
        SECOND_TRANSFORMER,
        "Macro F1:",
        round(second_result["best_macro_f1"], 4)
    )
else:
    print("Second transformer disabled.")


# 12. Optional multi-seed ensemble

If a single transformer is unstable, training the same architecture with different random seeds and averaging probabilities can improve robustness.

This is computationally expensive, so it is disabled by default.


In [ ]:
seed_results = []

if RUN_MULTIPLE_SEEDS:
    for seed in TRANSFORMER_SEEDS:
        if seed == SEED:
            seed_results.append(primary_result)
        else:
            result = train_transformer(
                model_name=TRANSFORMER_MODELS[PRIMARY_MODEL],
                seed=seed,
                max_length=MAX_LENGTH,
                learning_rate=LEARNING_RATE,
                epochs=EPOCHS,
            )
            seed_results.append(result)

    print("Trained seeds:", [r["seed"] for r in seed_results])
else:
    seed_results = [primary_result]
    print("Multiple seeds disabled.")


# 13. Build the best local probability ensemble

We search weights across the available model families.

Every model contributes **probabilities**, not hard 0/1 predictions.

This preserves more information and usually makes a better ensemble.


In [ ]:
available_models = [
    {
        "name": "TF-IDF",
        "prob": tfidf_valid_prob,
    }
]

for idx, result in enumerate(seed_results):
    available_models.append({
        "name": f"{PRIMARY_MODEL}_seed{result['seed']}",
        "prob": result["valid_prob"],
    })

if second_result is not None:
    available_models.append({
        "name": SECOND_TRANSFORMER,
        "prob": second_result["valid_prob"],
    })

print([m["name"] for m in available_models])


In [ ]:
# Pairwise ensemble search is deliberately simple and robust.
# The primary pair is TF-IDF + best transformer.
transformer_candidates = [
    m for m in available_models
    if "TF-IDF" not in m["name"]
]

best_transformer_candidate = max(
    transformer_candidates,
    key=lambda x: f1_score(
        y_valid,
        (
            x["prob"]
            >= tune_thresholds(
                y_valid,
                x["prob"]
            )[0].reshape(1, -1)
        ).astype(int),
        average="macro",
        zero_division=0,
    )
)

candidate_prob = best_transformer_candidate["prob"]

ensemble_rows = []
best_ensemble = None

for w in np.arange(0.0, 1.01, 0.02):
    prob = (
        w * tfidf_valid_prob
        + (1.0 - w) * candidate_prob
    )

    thresholds, threshold_df = tune_thresholds(
        y_valid,
        prob,
    )

    scores = multilabel_score(
        y_valid,
        prob,
        thresholds,
    )

    row = {
        "tfidf_weight": float(w),
        "transformer_weight": float(1.0 - w),
        "macro_f1": scores["macro_f1"],
        "micro_f1": scores["micro_f1"],
        "weighted_f1": scores["weighted_f1"],
        "samples_f1": scores["samples_f1"],
    }

    ensemble_rows.append(row)

    if (
        best_ensemble is None
        or row["macro_f1"] > best_ensemble["macro_f1"]
    ):
        best_ensemble = {
            **row,
            "prob": prob,
            "thresholds": thresholds,
            "threshold_df": threshold_df,
            "scores": scores,
            "transformer_name": best_transformer_candidate["name"],
        }

ensemble_search_df = pd.DataFrame(
    ensemble_rows
).sort_values(
    "macro_f1",
    ascending=False
)

display(ensemble_search_df.head(10).round(4))

print(
    "Best ensemble:",
    best_ensemble["tfidf_weight"],
    "TF-IDF +",
    best_ensemble["transformer_weight"],
    best_ensemble["transformer_name"]
)

print(
    "Macro F1:",
    round(best_ensemble["macro_f1"], 4)
)

display(
    best_ensemble["threshold_df"].round(4)
)


# 14. Local leaderboard

This lets you compare the main candidates before making the final CSV.


In [ ]:
leaderboard = [
    {
        "model": "TF-IDF",
        "macro_f1": tfidf_scores["macro_f1"],
        "micro_f1": tfidf_scores["micro_f1"],
    },
    {
        "model": PRIMARY_MODEL,
        "macro_f1": primary_scores["macro_f1"],
        "micro_f1": primary_scores["micro_f1"],
    },
    {
        "model": "TF-IDF + Transformer",
        "macro_f1": best_ensemble["macro_f1"],
        "micro_f1": best_ensemble["micro_f1"],
    },
]

if second_result is not None:
    second_scores = multilabel_score(
        y_valid,
        second_result["valid_prob"],
        second_result["thresholds"],
    )

    leaderboard.append({
        "model": SECOND_TRANSFORMER,
        "macro_f1": second_scores["macro_f1"],
        "micro_f1": second_scores["micro_f1"],
    })

leaderboard_df = pd.DataFrame(
    leaderboard
).sort_values(
    "macro_f1",
    ascending=False
)

display(leaderboard_df.round(4))


# 15. Error analysis for the selected ensemble

Before trusting the final submission, inspect difficult examples.


In [ ]:
final_valid_pred = best_ensemble["scores"]["predictions"]

error_df = valid_df[["ID", "text"]].copy()

for i, label in enumerate(LABELS):
    error_df[f"true_{label}"] = y_valid[:, i]
    error_df[f"pred_{label}"] = final_valid_pred[:, i]
    error_df[f"prob_{label}"] = best_ensemble["prob"][:, i]

wrong = (
    y_valid != final_valid_pred
).any(axis=1)

errors = error_df.loc[wrong].copy()

errors["wrong_label_count"] = (
    y_valid[wrong] != final_valid_pred[wrong]
).sum(axis=1)

print("Validation rows with at least one wrong label:", len(errors))

display(
    errors.sort_values(
        "wrong_label_count",
        ascending=False
    ).head(30)
)


# 16. Final training strategy

For the final CSV:

- TF-IDF is retrained on **train + validation**
- The selected transformer checkpoint is used for test prediction
- Validation-learned thresholds are preserved
- The locally selected ensemble weight is preserved

Why preserve thresholds?

Because the test labels are hidden. We cannot tune thresholds on the test set without leaking information.


In [ ]:
# ============================================================
# Final TF-IDF fit on train + validation
# ============================================================

full_labeled = pd.concat(
    [
        train_df[["clean_text"] + LABELS],
        valid_df[["clean_text"] + LABELS],
    ],
    ignore_index=True,
)

test_texts = test_df["clean_text"]

y_full = full_labeled[LABELS].to_numpy(dtype=np.int8)

final_word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=WORD_NGRAM_RANGE,
    min_df=2,
    max_df=0.995,
    max_features=WORD_MAX_FEATURES,
    sublinear_tf=True,
)

final_char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=CHAR_NGRAM_RANGE,
    min_df=2,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
)

X_full_word = final_word_vectorizer.fit_transform(
    full_labeled["clean_text"]
)
X_test_word = final_word_vectorizer.transform(
    test_texts
)

X_full_char = final_char_vectorizer.fit_transform(
    full_labeled["clean_text"]
)
X_test_char = final_char_vectorizer.transform(
    test_texts
)

X_full = hstack(
    [X_full_word, X_full_char],
    format="csr",
)

X_test = hstack(
    [X_test_word, X_test_char],
    format="csr",
)

final_tfidf_models = {}

for i, label in enumerate(LABELS):
    C = best_tfidf_C[label]

    clf = LogisticRegression(
        C=C,
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear",
        random_state=SEED,
    )

    clf.fit(
        X_full,
        y_full[:, i]
    )

    final_tfidf_models[label] = clf

test_tfidf_prob = np.column_stack([
    final_tfidf_models[label]
    .predict_proba(X_test)[:, 1]
    for label in LABELS
])

print("Test TF-IDF probability shape:", test_tfidf_prob.shape)


In [ ]:
# ============================================================
# Final transformer test prediction
# ============================================================

selected_transformer = best_ensemble["transformer_name"]

if selected_transformer.startswith(PRIMARY_MODEL):
    final_transformer_result = primary_result
elif (
    second_result is not None
    and selected_transformer == SECOND_TRANSFORMER
):
    final_transformer_result = second_result
else:
    final_transformer_result = primary_result

final_tokenizer = final_transformer_result["tokenizer"]
final_model = final_transformer_result["model"]

test_dataset = ToxicityDataset(
    test_df,
    final_tokenizer,
    MAX_LENGTH,
    labels=None,
)

test_collator = DataCollatorWithPadding(
    tokenizer=final_tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=VALID_BATCH_SIZE,
    shuffle=False,
    num_workers=2 if torch.cuda.is_available() else 0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=test_collator,
)

final_model.eval()
test_transformer_probs = []

with torch.no_grad():
    for batch in tqdm(
        test_loader,
        desc="Transformer test inference"
    ):
        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        logits = final_model(**batch).logits

        probs = torch.sigmoid(
            logits.float()
        ).cpu().numpy()

        test_transformer_probs.append(probs)

test_transformer_prob = np.vstack(
    test_transformer_probs
)

print(
    "Test transformer probability shape:",
    test_transformer_prob.shape
)


In [ ]:
# ============================================================
# Final ensemble + submission
# ============================================================

tfidf_weight = best_ensemble["tfidf_weight"]
transformer_weight = best_ensemble["transformer_weight"]

test_probability = (
    tfidf_weight * test_tfidf_prob
    + transformer_weight * test_transformer_prob
)

final_thresholds = best_ensemble["thresholds"]

test_prediction = (
    test_probability
    >= final_thresholds.reshape(1, -1)
).astype(np.int8)

submission = pd.DataFrame({
    "ID": test_df["ID"].values
})

for i, label in enumerate(LABELS):
    submission[label] = test_prediction[:, i]

submission = submission[
    ["ID"] + LABELS
]

assert len(submission) == len(test_df)
assert submission["ID"].tolist() == test_df["ID"].tolist()

for label in LABELS:
    assert set(
        submission[label].unique()
    ).issubset({0, 1})

SUBMISSION_PATH = Path(
    "submission_optimized_v2.csv"
)

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)

display(submission.head(20))

print(
    "Saved:",
    SUBMISSION_PATH.resolve()
)


# 17. Save everything needed for later experiments

The probability files are useful because you can later add another model without retraining every existing model.


In [ ]:
pd.DataFrame(
    best_ensemble["prob"],
    columns=[f"{x}_prob" for x in LABELS]
).assign(
    ID=valid_df["ID"].values
).to_csv(
    "validation_ensemble_probabilities.csv",
    index=False,
)

pd.DataFrame(
    test_probability,
    columns=[f"{x}_prob" for x in LABELS]
).assign(
    ID=test_df["ID"].values
).to_csv(
    "test_ensemble_probabilities.csv",
    index=False,
)

config = {
    "seed": SEED,
    "primary_model": PRIMARY_MODEL,
    "primary_model_name": TRANSFORMER_MODELS[PRIMARY_MODEL],
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "tfidf_C_per_label": best_tfidf_C,
    "tfidf_weight": float(tfidf_weight),
    "transformer_weight": float(transformer_weight),
    "selected_transformer": selected_transformer,
    "validation_macro_f1": float(best_ensemble["macro_f1"]),
    "thresholds": {
        label: float(t)
        for label, t in zip(
            LABELS,
            final_thresholds
        )
    },
}

with open(
    "optimized_v2_config.json",
    "w"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )

print(
    json.dumps(
        config,
        indent=2
    )
)


# 18. What to try if the first submission is not good enough

Do **not** randomly change everything.

Use this order:

### Experiment A — BanglaBERT learning rate

```text
1e-5
2e-5
3e-5
```

### Experiment B — sequence length

```text
128
192
256
```

### Experiment C — BanglishBERT

Set:

```python
RUN_SECOND_TRANSFORMER = True
```

### Experiment D — XLM-R

Change the second model to:

```python
SECOND_TRANSFORMER = "XLM-R"
```

### Experiment E — multiple seeds

```python
RUN_MULTIPLE_SEEDS = True
```

### Experiment F — class weighting

```python
USE_POS_WEIGHT = True
```

Do this last because class weighting can improve rare labels but can also reduce the overall Macro F1.


# 19. Final warning about score optimization

Never use the hidden Kaggle test labels to choose thresholds or models.

A safe workflow is:

```text
train
  ↓
validation
  ↓
choose model + thresholds
  ↓
test prediction
  ↓
submission.csv
  ↓
Kaggle score
```

If you repeatedly use the Kaggle score to make dozens of tiny changes, you can overfit the public leaderboard.

The goal is to find a model that generalizes.


In [ ]:
# 📥 Download the final submission CSV to your computer
from google.colab import files
import glob, os

candidates = [
    '/content/submission_optimized_v2.csv',
    '/content/data/submission_optimized_v2.csv'
] + glob.glob('/content/**/submission_optimized_v2.csv', recursive=True)

candidates = [p for p in candidates if os.path.isfile(p)]
assert candidates, 'Submission CSV was not found. Make sure the training/prediction cells finished successfully.'

submission_path = candidates[0]
print('Submission:', submission_path)
files.download(submission_path)
